# Airspeed velocity demo

In [ ]:
%%writefile asv.conf.json
{
  "version": 1,
  "project": "virtualenv",
  "repo": ".",
  "environment_type": "existing",
  "benchmark_dir": "benchmarks",
  "build_command": [],
  "matrix": {},
  "results_dir": "results"
}


Overwriting asv.conf.json


In [21]:
%%writefile benchmarks/bench_ops.py
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


import torch
from mypackage.silu import get_silu

def check_cuda_avail():
    if not torch.cuda.is_available():
        raise NotImplementedError(
            "Torch could not detect your GPU, so..."
        )


class TimeFusionCUDA:
    params = [1024, 4096]
    param_names = ["size"]

    def setup(self, size):
        check_cuda_avail()

        self.x = torch.randn(size, size, device="cuda")
        # compile if necessary
        self.silu = get_silu()
        # warmup (avoid measuring compile)
        self.silu(self.x)
        torch.cuda.synchronize()

    def time_unfused(self, size):
        self.silu(self.x)
        torch.cuda.synchronize()


class MemFusionCUDA:
    params = [1024, 4096]
    param_names = ["size"]

    def setup(self, size):
        check_cuda_avail()
        
        self.x = torch.randn(size, size, device="cuda")
        self.silu = get_silu()
        self.silu(self.x)
        torch.cuda.synchronize()

    def peakmem_unfused(self, size):
        self.silu(self.x)
        torch.cuda.synchronize()


Overwriting benchmarks/bench_ops.py


In [ ]:
%%writefile mypackage/silu.py
import torch
import typing as tp

def get_silu() -> tp.Callable[[torch.Tensor], torch.Tensor]:
    def fn(x):
        return x * x.sigmoid()
    return fn

Overwriting src/mypackage/silu.py


In [15]:
%%bash

git add src
git commit -am 'Silly silu'

[master 96814d2] Silly silu
 2 files changed, 1 insertion(+), 4 deletions(-)


In [16]:
%%bash
uv run asv run --python=same --set-commit-hash=$(git rev-parse HEAD)

· Discovering benchmarks
· Running 2 total benchmarks (1 commits * 1 environments * 2 benchmarks)
[ 0.00%] · For virtualenv commit 96814d29 <master>:
[ 0.00%] ·· Building for existing-py_home_yotterni_asv_demo_.venv_bin_python3
[ 0.00%] ·· Benchmarking existing-py_home_yotterni_asv_demo_.venv_bin_python3
[50.00%] ··· Running (bench_ops.TimeFusionCUDA.time_unfused--).
[75.00%] ··· bench_ops.MemFusionCUDA.peakmem_unfused                         ok
[75.00%] ··· ====== ======
              size        
             ------ ------
              1024   468M 
              4096   468M 
             ====== ======

[100.00%] ··· bench_ops.TimeFusionCUDA.time_unfused                           ok
[100.00%] ··· ====== ===========
               size             
              ------ -----------
               1024   413±100μs 
               4096   877±200μs 
              ====== ===========



In [17]:
%%writefile src/mypackage/silu.py
import torch
import typing as tp

def get_silu() -> tp.Callable[[torch.Tensor], torch.Tensor]:
    def fn(x):
        return x * x.sigmoid()

    return torch.compile(fn)

Overwriting src/mypackage/silu.py


In [18]:
%%bash

git add mypackage
git commit -am 'Compiled silu'

fatal: pathspec 'mypackage' did not match any files


[master fcebf14] Compiled silu
 1 file changed, 2 insertions(+), 1 deletion(-)


In [19]:
%%bash
uv run asv run --python=same --set-commit-hash=$(git rev-parse HEAD)

· Discovering benchmarks
· Running 2 total benchmarks (1 commits * 1 environments * 2 benchmarks)
[ 0.00%] · For virtualenv commit fcebf148 <master>:
[ 0.00%] ·· Building for existing-py_home_yotterni_asv_demo_.venv_bin_python3
[ 0.00%] ·· Benchmarking existing-py_home_yotterni_asv_demo_.venv_bin_python3
[50.00%] ··· Running (bench_ops.TimeFusionCUDA.time_unfused--).
[75.00%] ··· bench_ops.MemFusionCUDA.peakmem_unfused                         ok
[75.00%] ··· ====== ======
              size        
             ------ ------
              1024   653M 
              4096   653M 
             ====== ======

[100.00%] ··· bench_ops.TimeFusionCUDA.time_unfused                           ok
[100.00%] ··· ====== ===========
               size             
              ------ -----------
               1024   60.9±60μs 
               4096    417±20μs 
              ====== ===========



In [20]:
%%bash

uv run asv publish
uv run asv preview

[11.11%] · Loading machine info
[22.22%] · Getting params, commits, tags and branches
[33.33%] · Loading results
[44.44%] · Detecting steps
[55.56%] · Generating graphs
[66.67%] · Generating output for SummaryGrid
[77.78%] · Generating output for SummaryList
[88.89%] · Generating output for Regressions
[100.00%] · Writing index
· Serving at http://127.0.0.1:8080/
· Press ^C to abort
  


127.0.0.1 - - [17/Mar/2026 21:53:49] "GET /info.json?_=1773773628897 HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 21:53:49] "GET /index.json?timestamp=1773773622709 HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 21:53:49] "GET /graphs/summary/bench_ops.MemFusionCUDA.peakmem_unfused.json?timestamp=1773773622709 HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 21:53:49] "GET /graphs/summary/bench_ops.TimeFusionCUDA.time_unfused.json?timestamp=1773773622709 HTTP/1.1" 200 -
127.0.0.1 - - [17/Mar/2026 21:53:51] "GET /graphs/arch-x86_64/branch-master/cpu-Intel(R)%20Xeon(R)%20Gold%206230R%20CPU%20%40%202.10GHz/machine-Sydney/num_cpu-104/os-Linux%205.15.0-139-generic/python-_home_yotterni_asv_demo_.venv_bin_python3/ram-1056469076/bench_ops.TimeFusionCUDA.time_unfused.json?timestamp=1773773622709 HTTP/1.1" 200 -


TypeError: %d format: a real number is required, not NoneType

TypeError: %d format: a real number is required, not NoneType

In [ ]:
# %%bash

# source .venv/bin/activate
# asv run HEAD~1..HEAD -vvv
# asv run ALL

In [ ]:
# %%bash

# source .venv/bin/activate
# uv run asv publish
# uv run asv preview